In [0]:
%pip install genanki

In [0]:
import html
import re
import shutil

import genanki

GOLD_TABLE = "workspace.linguacopilot.gold_flashcards"
EXPORT_VOLUME = "/Volumes/workspace/linguacopilot/exports"
LOCAL_FILE = "/tmp/linguacopilot_de.apkg"
DECK_NAME = "LinguaCopilot::Allemand"

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.linguacopilot.exports")

# Fixed ids so that Anki recognizes the same deck and note type at each import
MODEL_ID = 1607392322
DECK_ID = 2059400110

CARD_CSS = """
.card {
  font-family: Roboto, Arial, sans-serif;
  background: #ffffff;
  color: #0f0f0f;
  text-align: center;
  padding: 24px 12px;
}
.card.nightMode, .nightMode .card {
  background: #0f0f0f;
  color: #f1f1f1;
}

/* Light frame with a red progress bar, like the YouTube player */
.lc-player {
  position: relative;
  overflow: hidden;
  max-width: 720px;
  margin: 0 auto;
  padding: 44px 24px 36px;
  border: 1px solid #e5e5e5;
  border-radius: 12px;
  background: #f9f9f9;
}
.lc-player::after {
  content: "";
  position: absolute;
  left: 0;
  bottom: 0;
  width: 100%;
  height: 5px;
  background: linear-gradient(90deg, #ff0033 0%, #ff0033 40%, #e5e5e5 40%, #e5e5e5 100%);
}
.nightMode .lc-player {
  background: #212121;
  border-color: #303030;
}

/* German sentence styled like a YouTube subtitle (direct child only) */
.lc-caption > span {
  background: rgba(8, 8, 8, 0.75);
  color: #ffffff;
  font-size: 26px;
  line-height: 1.7;
  padding: 3px 8px;
  -webkit-box-decoration-break: clone;
  box-decoration-break: clone;
}
.lc-caption .cloze {
  color: #ff4e45;
  font-weight: 700;
}

/* French translation as a second, lighter line */
.lc-subcaption {
  margin-top: 14px;
  color: #606060;
  font-size: 17px;
}
.nightMode .lc-subcaption {
  color: #aaaaaa;
}

/* Answer field shaped like the YouTube search bar */
#typeans {
  display: block;
  width: 100%;
  max-width: 720px;
  box-sizing: border-box;
  margin: 24px auto 0;
  padding: 12px 20px;
  border: 2px solid #e5e5e5;
  border-radius: 40px;
  font-family: Roboto, Arial, sans-serif;
  font-size: 20px;
  text-align: center;
  outline: none;
}
#typeans:focus {
  border-color: #ff0033;
}
.nightMode #typeans {
  background: #121212;
  border-color: #303030;
  color: #f1f1f1;
}
.typeGood { background: #2ba640; color: #ffffff; }
.typeBad { background: #ff4e45; color: #ffffff; }
.typeMissed { background: #717171; color: #ffffff; }

/* Article and plural, shown on the back only */
.lc-chip {
  display: inline-block;
  margin-top: 20px;
  padding: 8px 16px;
  border-radius: 8px;
  background: #ffe5e9;
  color: #cc0029;
  font-size: 18px;
  font-weight: 500;
}
.nightMode .lc-chip {
  background: #3d1016;
  color: #ff8a95;
}

/* YouTube link as a red play button */
.lc-source a {
  display: inline-block;
  margin-top: 20px;
  padding: 10px 18px;
  border-radius: 18px;
  background: #ff0033;
  color: #ffffff;
  font-size: 15px;
  font-weight: 600;
  text-decoration: none;
}
.lc-source a::before {
  content: "▶  ";
}
"""

FRONT = """
<div class="lc-player">
  <div class="lc-caption"><span>{{cloze:Text}}</span></div>
  <div class="lc-subcaption">{{Context}}</div>
</div>
{{type:cloze:Text}}
"""

BACK = """
<div class="lc-player">
  <div class="lc-caption"><span>{{cloze:Text}}</span></div>
  <div class="lc-subcaption">{{Context}}</div>
</div>
{{type:cloze:Text}}
{{#Grammar}}<div class="lc-chip">{{Grammar}}</div>{{/Grammar}}
<div class="lc-source">{{Source}}</div>
"""

model = genanki.Model(
    MODEL_ID,
    "LinguaCopilot Cloze (type answer) v4",
    fields=[{"name": "Text"}, {"name": "Context"}, {"name": "Grammar"}, {"name": "Source"}],
    templates=[{"name": "Cloze", "qfmt": FRONT, "afmt": BACK}],
    css=CARD_CSS,
    model_type=genanki.Model.CLOZE,
)

deck = genanki.Deck(DECK_ID, DECK_NAME)

for row in spark.table(GOLD_TABLE).collect():
    lemma = html.escape(row.lemma)
    translation = html.escape(row.translation)
    word = html.escape(row.example_word_form)
    sentence = html.escape(row.example_sentence)
    source = f'<a href="{row.youtube_url}">Voir le passage sur YouTube</a>' if row.youtube_url else ""

    # Front: the YouTube sentence with the word as a cloze, translation as hint
    cloze = f"{{{{c1::{word}::{translation}}}}}"
    text, count = re.subn(rf"\b{re.escape(word)}\b", cloze, sentence, count=1)
    if count == 0:
        print(f"Skipped {row.lemma}: word not found in its sentence")
        continue

    # Back only: article and plural for nouns
    grammar = ""
    if row.article:
        grammar = f"{row.article} {lemma}"
        if row.plural:
            grammar += f", pluriel die {html.escape(row.plural)}"

    # Stable id: re-importing updates the card instead of creating a duplicate
    guid = genanki.guid_for("linguacopilot", "word", row.lemma)
    deck.add_note(
        genanki.Note(
            model=model,
            fields=[text, html.escape(row.example_sentence_translation), grammar, source],
            guid=guid,
        )
    )

genanki.Package(deck).write_to_file(LOCAL_FILE)
shutil.copy(LOCAL_FILE, f"{EXPORT_VOLUME}/linguacopilot_de.apkg")
print(f"{len(deck.notes)} notes exported to {EXPORT_VOLUME}/linguacopilot_de.apkg")